In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv('heart_cleaned.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,ca,thal,target
0,67,1,0,176,148,1,1,117,1,3.1,3,2,0
1,57,1,1,155,551,0,1,127,0,2.5,3,1,0
2,43,1,0,125,519,1,1,188,0,3.3,0,1,0
3,71,1,0,123,285,0,0,115,0,3.7,0,2,0
4,36,0,0,122,488,1,1,74,0,0.1,2,0,1


In [3]:
df.shape

(500, 13)

In [4]:
df.isnull().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
ca          0
thal        0
target      0
dtype: int64

In [5]:
df.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,ca,thal,target
count,500.000000,500.000000,500.00000,500.00000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,52.980000,0.468000,1.44600,145.80200,349.134000,0.522000,0.516000,138.350000,0.476000,3.111400,1.474000,1.034000,0.500000
std,13.800598,0.499475,1.09795,30.52058,130.505891,0.500016,0.500244,37.976254,0.499924,1.810405,1.115263,0.813737,0.500501
min,29.000000,0.000000,0.00000,94.00000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,42.000000,0.000000,0.00000,119.00000,240.500000,0.000000,0.000000,108.000000,0.000000,1.600000,0.000000,0.000000,0.000000
50%,54.000000,0.000000,1.00000,146.00000,351.500000,1.000000,1.000000,139.000000,0.000000,3.100000,1.000000,1.000000,0.500000
75%,64.000000,1.000000,2.00000,174.00000,463.500000,1.000000,1.000000,172.000000,1.000000,4.625000,2.000000,2.000000,1.000000
max,76.000000,1.000000,3.00000,199.00000,563.000000,1.000000,1.000000,201.000000,1.000000,6.200000,3.000000,2.000000,1.000000


In [6]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [7]:
models = {
    'SVM': SVC(kernel='rbf', C=1.0),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

In [8]:
results = {}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results[name] = [acc, prec, rec, f1]
    print(name, acc, prec, rec, f1)

SVM 0.83 0.8163265306122449 0.8333333333333334 0.8247422680412371
Logistic Regression 0.88 0.8913043478260869 0.8541666666666666 0.8723404255319149
Random Forest 0.83 0.8163265306122449 0.8333333333333334 0.8247422680412371
KNN 0.83 0.803921568627451 0.8541666666666666 0.8282828282828283


In [9]:
res_df = pd.DataFrame(results, index=['accuracy','precision','recall','f1']).T
res_df = res_df.sort_values('accuracy', ascending=False)
res_df

,accuracy,precision,recall,f1
Logistic Regression,0.88,0.891304,0.854167,0.872340
SVM,0.83,0.816327,0.833333,0.824742
Random Forest,0.83,0.816327,0.833333,0.824742
KNN,0.83,0.803922,0.854167,0.828283


In [10]:
best_model = res_df['accuracy'].idxmax()
print('best model is', best_model)
print(res_df.loc[best_model])

best model is Logistic Regression
accuracy     0.880000
precision    0.891304
recall       0.854167
f1           0.872340
Name: Logistic Regression, dtype: float64


In [11]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():
    scores = cross_val_score(model, X_train_s, y_train, cv=5)
    print(name, scores.mean(), scores.std())

SVM 0.8300000000000001 0.03758324094593227
Logistic Regression 0.8300000000000001 0.012747548783981965


Random Forest 0.805 0.029154759474226508
KNN 0.765 0.0508674748734395
